In [ ]:
!pip install sentence-transformers faiss-cpu numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import time
import numpy as np
import faiss

BASE = '/content/drive/MyDrive/semantic-search-system'

with open(f'{BASE}/data/processed/clean_corpus.pkl', 'rb') as f:
    corpus = pickle.load(f)

texts = corpus['texts']
print(f'Loaded {len(texts)} documents from Drive')

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'BAAI/bge-base-en-v1.5'
print(f'Loading model: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME)
DIM = model.get_sentence_embedding_dimension()
print(f'Embedding dimension: {DIM}')

In [ ]:
print(f'Encoding {len(texts)} documents...')
start = time.time()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(f'Done in {time.time()-start:.1f}s')
print(f'Shape: {embeddings.shape}')

In [ ]:
index = faiss.IndexFlatIP(DIM)
index.add(embeddings.astype(np.float32))
print(f'FAISS index built with {index.ntotal} vectors')

In [ ]:
np.save(f'{BASE}/models/embeddings.npy', embeddings)
faiss.write_index(index, f'{BASE}/models/faiss.index')

print(f'Saved: models/embeddings.npy')
print(f'Saved: models/faiss.index')
print(f'Sizes:')
print(f'  embeddings.npy : {os.path.getsize(f"{BASE}/models/embeddings.npy")/1e6:.1f} MB')
print(f'  faiss.index    : {os.path.getsize(f"{BASE}/models/faiss.index")/1e6:.1f} MB')

In [ ]:
test_queries = [
    'symptoms of flu and fever medication',
    'space shuttle launch nasa',
    'graphics card driver installation windows',
]

for q in test_queries:
    q_emb = model.encode([q], normalize_embeddings=True).astype(np.float32)
    scores, idxs = index.search(q_emb, k=3)
    print(f'\nQuery: {q}')
    for rank, (idx, score) in enumerate(zip(idxs[0], scores[0]), 1):
        print(f'  #{rank} score={score:.3f} [{corpus["categories"][idx]}]')